# 📊 View — Top Categorias

Validação da view `vw_top_categorias` antes de mover para o Streamlit.

In [2]:
import pandas as pd
import numpy as np
import sys
sys.path.append('..')

#formata todos os números float com 2 casas decimais na exibição do Jupyter.
pd.set_option('display.float_format', '{:.2f}'.format)

pedidos    = pd.read_csv("../dados/pedidos_limpo.csv", parse_dates=[
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
])

pagamentos = pd.read_csv("../dados/pagamentos_limpo.csv")
itens      = pd.read_csv("../dados/itens_limpo.csv", parse_dates=['shipping_limit_date'])
produtos   = pd.read_csv("../dados/produtos_limpo.csv")

print("Dados carregados!")


Dados carregados!


## 🧪 Testando o código antes de criar a view

In [2]:
pagamentos.head(2)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39


In [3]:
itens.head(2)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93


In [4]:
produtos.head(1)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.00,287.00,1,225.00,16.00,10.00,14.00


In [3]:

# Join entre itens, produtos, pedidos e pagamentos
df = (itens
      .merge(produtos[['product_id', 'product_category_name']], on='product_id', how='left')
      .merge(pedidos[['order_id', 'order_status', 'order_purchase_timestamp']], on='order_id', how='left')
      .merge(pagamentos, on='order_id', how='left'))

# Filtrando só pedidos entregues
df = df[df['order_status'] == 'delivered']

df.head(2)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,order_status,order_purchase_timestamp,payment_sequential,payment_type,payment_installments,payment_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,delivered,2017-09-13 08:59:02,1.00,credit_card,2.00,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,delivered,2017-04-26 10:53:06,1.00,credit_card,3.00,259.83


In [4]:
df = (itens
          .merge(produtos[['product_id', 'product_category_name']], on='product_id', how='left')
          .merge(pedidos[['order_id', 'order_status', 'order_purchase_timestamp']], on='order_id', how='left'))

df = df[df['order_status'] == 'delivered']

df['ano']             = df['order_purchase_timestamp'].dt.year
df['data_mes']        = df['order_purchase_timestamp'].dt.to_period('M').astype(str)
df['receita_produto'] = df['price']
df['receita_frete']   = df['freight_value']
df['receita_item']    = df['price'] + df['freight_value']

top_categorias = (df.groupby(['product_category_name', 'ano', 'data_mes'])
                   .agg(
                       total_pedidos    = ('order_id',        'nunique'),
                       total_itens      = ('order_item_id',   'count'),
                       total_vendedores = ('seller_id',       'nunique'),
                       receita_produtos = ('receita_produto', 'sum'),
                       receita_frete    = ('receita_frete',   'sum'),
                       receita_total    = ('receita_item',    'sum'),
                   )
                   .reset_index()
                   .sort_values('receita_total', ascending=False))

top_categorias['receita_total']    = top_categorias['receita_total'].round(2)
top_categorias['receita_produtos'] = top_categorias['receita_produtos'].round(2)
top_categorias['receita_frete']    = top_categorias['receita_frete'].round(2)
top_categorias['ticket_medio']     = (top_categorias['receita_total'] / top_categorias['total_pedidos']).round(2)
top_categorias['preco_medio_item'] = (top_categorias['receita_produtos'] / top_categorias['total_itens']).round(2)

top_categorias.head(10)

,product_category_name,ano,data_mes,total_pedidos,total_itens,total_vendedores,receita_produtos,receita_frete,receita_total,ticket_medio,preco_medio_item
231,beleza_saude,2018,2018-08,765,835,176,119391.01,16253.50,135644.51,177.31,142.98
1169,relogios_presentes,2018,2018-05,585,625,40,119364.98,8612.42,127977.40,218.76,190.98
229,beleza_saude,2018,2018-06,790,875,170,106745.72,17112.25,123857.97,156.78,122.00
230,beleza_saude,2018,2018-07,699,770,169,103524.22,16700.82,120225.04,172.00,134.45
801,informatica_acessorios,2018,2018-02,791,971,81,100131.99,17084.66,117216.65,148.19,103.12
228,beleza_saude,2018,2018-05,669,751,145,94534.38,13726.98,108261.36,161.83,125.88
264,cama_mesa_banho,2017,2017-11,804,961,55,87957.63,17005.95,104963.58,130.55,91.53
1171,relogios_presentes,2018,2018-07,507,521,36,95164.79,9737.71,104902.50,206.91,182.66
227,beleza_saude,2018,2018-04,619,685,140,91058.45,13274.45,104332.90,168.55,132.93
1167,relogios_presentes,2018,2018-03,407,421,35,95661.18,8631.08,104292.26,256.25,227.22


### IMPORTANDO VIEW

In [6]:
from views.vw_top_categorias import get_top_categorias

df_categorias = get_top_categorias(pedidos, itens, produtos)
df_categorias.head()

,product_category_name,ano,data_mes,total_pedidos,total_itens,total_vendedores,receita_produtos,receita_frete,receita_total,ticket_medio,preco_medio_item
231,beleza_saude,2018,2018-08,765,835,176,119391.01,16253.50,135644.51,177.31,142.98
1169,relogios_presentes,2018,2018-05,585,625,40,119364.98,8612.42,127977.40,218.76,190.98
229,beleza_saude,2018,2018-06,790,875,170,106745.72,17112.25,123857.97,156.78,122.00
230,beleza_saude,2018,2018-07,699,770,169,103524.22,16700.82,120225.04,172.00,134.45
801,informatica_acessorios,2018,2018-02,791,971,81,100131.99,17084.66,117216.65,148.19,103.12
